# Data Cleaning: Football Stadium Attendance Dataset

In this notebook we take `data/full_dataset.csv` (the raw pull from CollegeFootballData and Open-Meteo, see `script/pull_full_data.py`) and clean it before we start building our attendance classifier. Our goal here is not to build the model yet. It is to make sure the data we hand off is complete, trustworthy, and that every decision we made about it is explained.

For every issue we found, we followed the same process: figure out why it is happening, decide whether it is something we can safely fix, and document the reasoning so the rest of the team, and anyone reviewing this, can see why we made that call instead of a different one. The cleaned result is saved to `data/full_dataset_clean.csv` at the end.

In [33]:
import pandas as pd

df = pd.read_csv("../data/full_dataset.csv")
print(df.shape)
df.head()

(2021, 56)


,id,season,week,seasonType,startDate,startTimeTBD,completed,neutralSite,conferenceGame,attendance,...,elevation,constructionYear,game_date,temperature_c,precip_mm,windspeed_kmh,attendance_pct,high_attendance,flag_missing_attendance,flag_missing_capacity
0,401635525,2024,1,regular,2024-08-24T16:00:00.000Z,False,True,True,True,47998.0,...,2.264277,2010.0,2024-08-24,17.1,0.0,22.8,0.928395,NaN,False,False
1,401643697,2024,1,regular,2024-08-24T20:00:00.000Z,False,True,False,False,17314.0,...,1554.471924,1960.0,2024-08-24,30.6,0.0,2.6,0.441413,0.0,False,False
2,401643696,2024,1,regular,2024-08-25T00:00:00.000Z,False,True,False,False,20263.0,...,1406.576050,1966.0,2024-08-25,15.1,0.0,11.7,0.750481,0.0,False,False
3,401643858,2024,1,regular,2024-08-25T03:59:00.000Z,False,True,False,False,12206.0,...,20.000000,2015.0,2024-08-25,26.8,0.1,33.0,0.803343,0.0,False,False
4,401628458,2024,1,regular,2024-08-29T22:00:00.000Z,False,True,False,False,47803.0,...,19.000000,1994.0,2024-08-29,23.4,0.0,16.1,0.911332,1.0,False,False


## 1. Missing Value Assessment

Before deciding how to fix anything, we want to see the full picture of what is empty in the dataset. We also check for duplicate rows or duplicate game IDs, since those would be a much bigger problem than missing values.

In [34]:
missing = df.isna().mean().sort_values(ascending=False)
missing[missing > 0].to_frame("pct_missing").style.format("{:.1%}")

,pct_missing
highlights,100.0%
playoff,100.0%
notes,98.4%
awayPostgameElo,17.3%
awayPregameElo,17.3%
high_attendance,17.2%
attendance_pct,14.3%
attendance,14.2%
zip,0.2%
constructionYear,0.2%


In [35]:
print("duplicate rows:", df.duplicated().sum())
print("duplicate game ids:", df["id"].duplicated().sum())

duplicate rows: 0
duplicate game ids: 0


## 2. Removal of Non-Informative Columns

A few columns turned out to be dead weight once we looked closer:

- `highlights` and `playoff` are 100% empty, so there is nothing to work with.
- `notes` is 98%+ empty and free text anyway, not something we would use as a model feature.
- `id_venue` and `name` turned out to be exact duplicates of `venueId` and `venue`, a leftover from how the venue data was merged in (`suffixes=("", "_venue")` in the pull script).
- `seasonType`, `completed`, and `homeClassification` only ever have one value, since the pull script already filtered on them (`"regular"`, `True`, `"fbs"`). A column that never changes cannot help a model tell games apart.

We check each of these assumptions with an `assert` before dropping anything, so we are not deleting a column based on a guess.

In [36]:
assert (df["venueId"] == df["id_venue"]).all()
assert (df["venue"] == df["name"]).all()
assert df["seasonType"].nunique() == 1
assert df["completed"].nunique() == 1
assert df["homeClassification"].nunique() == 1

df = df.drop(columns=[
    "highlights", "playoff", "notes",
    "id_venue", "name",
    "seasonType", "completed", "homeClassification",
])
df.shape

(2021, 48)

## 3. Explanation of Missing Label Values

`attendance_pct` and `high_attendance`, our actual prediction target, are `NaN` for a chunk of rows, and at first that looked concerning. Looking at `add_label()` in `script/pull_full_data.py`, the label is only supposed to be defined when three things are all true: the game is not at a neutral site, the attendance is known, and the venue's capacity is known. If any of those is missing, there is no way to compute a meaningful attendance percentage, so the label is correctly left blank.

We wanted to prove that this is really what is happening rather than just assume it, so below we check that every row with a missing label falls into one of those three categories. If any row fell outside all three, that would mean something is actually broken and we would need to investigate further. The `assert` at the end makes sure that never happens silently.

In [37]:
print("neutral-site games:", df["neutralSite"].sum())
print("missing attendance:", df["flag_missing_attendance"].sum())
print("missing capacity:  ", df["flag_missing_capacity"].sum())
print("unlabelable (high_attendance NaN):", df["high_attendance"].isna().sum())

# every unlabelable row should be explained by one of: neutral site, missing attendance, missing capacity
unexplained = df[
    df["high_attendance"].isna()
    & ~df["neutralSite"]
    & ~df["flag_missing_attendance"]
    & ~df["flag_missing_capacity"]
]
assert unexplained.empty, f"{len(unexplained)} rows missing a label for no known reason"

neutral-site games: 68
missing attendance: 286
missing capacity:   4
unlabelable (high_attendance NaN): 348


## 4. Handling of Over-Capacity Attendance Records

Two data quality issues came up when our team reviewed the dataset together:

- **14.2% of games are missing attendance.** CFBD simply does not have the number for every game. There is nothing we can do about that except be aware of it (already tracked with `flag_missing_attendance`).
- **17.9% of labelable games show attendance above 100% of listed capacity.** Our best explanation is that CFBD's `capacity` field reflects the stadium's current size, not what it was in the season the game was actually played, since stadiums get renovated over time. This is not a data entry error, but it is still something we needed to decide how to handle, since it directly affects our label.

We considered three options: cap `attendance_pct` at 1.0, exclude these games entirely, or leave the numbers as they are. We chose to leave them as is, uncapped and not excluded, and instead of just asserting that this is fine, we proved it below. Every one of these over-capacity games that is actually labelable is already scored `high_attendance = 1`, because any percentage at or above 1.0 automatically clears our 0.90 cutoff. So capping to 1.0 would not change a single label. It would only remove information. Excluding these rows would instead mean deleting real, legitimately full games, which could bias our dataset toward whichever stadiums happen to have outdated capacity numbers on record. We kept a `flag_over_capacity` column so this can still be filtered on if a different part of the analysis needs to treat them differently.

In [38]:
df["flag_over_capacity"] = df["attendance_pct"] > 1.0

n_labelable = df["attendance_pct"].notna().sum()
n_over = df["flag_over_capacity"].sum()
print(f"over-capacity: {n_over} of {n_labelable} labelable games ({n_over / n_labelable:.1%})")

df[df["flag_over_capacity"]][["season", "homeTeam", "venue", "attendance", "capacity", "attendance_pct"]].sort_values(
    "attendance_pct", ascending=False
).head(10)

over-capacity: 308 of 1731 labelable games (17.8%)


,season,homeTeam,venue,attendance,capacity,attendance_pct
1106,2025,Louisiana Tech,Joe Aillet Stadium,60229.0,28019.0,2.149577
1027,2025,Charlotte,Jerry Richardson Stadium,19233.0,15300.0,1.257059
996,2025,UTSA,Alamodome,45778.0,36582.0,1.251380
48,2024,App State,Kidd Brewer Stadium,36232.0,30000.0,1.207733
1266,2025,App State,Kidd Brewer Stadium,35021.0,30000.0,1.167367
600,2024,App State,Kidd Brewer Stadium,34954.0,30000.0,1.165133
1000,2025,App State,Kidd Brewer Stadium,34921.0,30000.0,1.164033
180,2024,Kansas,Children's Mercy Park,21493.0,18467.0,1.163860
692,2024,Navy,Navy-Marine Corps Memorial Stadium,38914.0,34000.0,1.144529
548,2024,Toledo,Glass Bowl,29697.0,26038.0,1.140525


In [39]:
HIGH_CUTOFF = 0.90  # must match script/pull_full_data.py

labelable_over_capacity = df[df["flag_over_capacity"] & df["high_attendance"].notna()]
print(f"labelable over-capacity games: {len(labelable_over_capacity)}")
print(f"of those, already labeled high_attendance=1: {(labelable_over_capacity['high_attendance'] == 1).sum()}")

# proves capping at 1.0 is a no-op for the label at this cutoff
assert (labelable_over_capacity["high_attendance"] == 1).all()
assert HIGH_CUTOFF <= 1.0

labelable over-capacity games: 299
of those, already labeled high_attendance=1: 299


## 5. Investigation of Missing Away-Team Elo Ratings

The pull script filters the home team down to FBS only, but does not do the same for the away team. That means an FCS opponent can appear in the data, and CFBD simply does not compute Elo ratings for FCS teams. We check the away team classification for every row with a missing Elo to confirm that this is really the full explanation.

In [40]:
missing_away_elo = df[df["awayPregameElo"].isna()]
print(missing_away_elo["awayClassification"].value_counts())

awayClassification
fcs    350
Name: count, dtype: int64


Confirmed: every single missing away Elo belongs to an FCS opponent. So this is not random noise, it is a whole population CFBD never rates.

For how to fill it in, we did not want to use the same generic number everywhere, since that would treat a genuinely strong FCS program the same as a weak one. Instead, our approach is: if a team has any rated games elsewhere in the dataset, use that team's own average Elo to fill its missing games. Most teams (106 of 109) never have a rated game at all, so there is nothing team specific to fall back on. For those, we use the overall median Elo instead. Either way, we also add an `awayEloMissing` flag column, so the model can learn that a game had an unrated opponent as its own signal, instead of just trusting whatever number we filled in.

In [41]:
df["awayEloMissing"] = df["awayPregameElo"].isna()

for col in ["awayPregameElo", "awayPostgameElo"]:
    team_mean = df.groupby("awayTeam")[col].transform("mean")
    global_median = df[col].median()
    n_before = df[col].isna().sum()
    df[col] = df[col].fillna(team_mean).fillna(global_median)
    n_team_filled = (df["awayEloMissing"] & team_mean.notna()).sum()
    print(f"{col}: {n_before} missing, {n_team_filled} filled from team's own mean, "
          f"{n_before - n_team_filled} filled from global median ({global_median:.1f})")

assert df["awayPregameElo"].isna().sum() == 0
assert df["awayPostgameElo"].isna().sum() == 0

awayPregameElo: 350 missing, 5 filled from team's own mean, 345 filled from global median (1480.0)
awayPostgameElo: 350 missing, 5 filled from team's own mean, 345 filled from global median (1472.0)


## 5b. Home-Team Elo Missing for One Record

We almost missed this one, since it rounds to 0.0% missing. `homePregameElo` and `homePostgameElo` are blank for exactly one row: UNLV playing at home in week 1 of the 2025 season. This cannot be the same FCS explanation as above, since every home team in this dataset is FBS. Looking at UNLV's full record, it has an Elo rating for 12 of its 13 games, just not the very first game of a new season. Our best explanation is that CFBD had not carried the rating forward from the previous season yet at the time we pulled the data, a timing gap rather than a "this team is not rated" situation.

Since UNLV has plenty of its own history to draw from, the same team mean approach from section 5 fixes this one completely on its own, with no need for the median fallback.

In [42]:
df["homeEloMissing"] = df["homePregameElo"].isna()

for col in ["homePregameElo", "homePostgameElo"]:
    team_mean = df.groupby("homeTeam")[col].transform("mean")
    global_median = df[col].median()
    n_before = df[col].isna().sum()
    df[col] = df[col].fillna(team_mean).fillna(global_median)
    print(f"{col}: {n_before} missing, filled from team's own mean")

assert df["homePregameElo"].isna().sum() == 0
assert df["homePostgameElo"].isna().sum() == 0

homePregameElo: 1 missing, filled from team's own mean
homePostgameElo: 1 missing, filled from team's own mean


## 5c. Missing Postgame Win Probability and Excitement Index

`homePostgameWinProbability`, `awayPostgameWinProbability`, and `excitementIndex` are all blank for the exact same game: Miami (OH) at Kent State, 2024-11-14. This looks like a one-off gap on CFBD's end. There is no other row for this specific game we could use to fill it in, so we are leaving it as `NaN` rather than making up a number.

Something worth flagging for whoever builds the model next, separate from this missing value: these three columns are all postgame statistics. They only exist once the final score is known. Since our whole goal is predicting attendance ahead of time, none of these should be used as model inputs anyway, missing or not, since using them would mean feeding the model information from the future.

## 6. Missing Venue Details

Tracking down which rows are missing venue information, it turns out only 3 distinct venues have any gap at all: Aviva Stadium (Dublin), Wembley Stadium (London), and Wrigley Field (Chicago).

- Aviva and Wembley are missing `state` and `zip`, which makes sense once we notice they are outside the United States. That is not a data problem, it is correctly blank.
- Wrigley Field is missing `grass`, `timezone`, `elevation`, and `constructionYear`, and this one is a real gap, since Wrigley Field is a well known, well documented venue. Since this affects only a few rows, we looked up the real facts ourselves rather than trying to statistically estimate them.

In [43]:
venue_cols = ["grass", "state", "zip", "timezone", "elevation", "constructionYear"]
df[df[venue_cols].isna().any(axis=1)][["venue", "city", "countryCode"] + venue_cols]

,venue,city,countryCode,grass,state,zip,timezone,elevation,constructionYear
0,Aviva Stadium,Dublin,IE,True,NaN,NaN,Europe/Dublin,2.264277,2010.0
689,Wrigley Field,Chicago,US,NaN,IL,60654.0,NaN,NaN,NaN
820,Wrigley Field,Chicago,US,NaN,IL,60654.0,NaN,NaN,NaN
873,Aviva Stadium,Dublin,IE,True,NaN,NaN,Europe/Dublin,2.264277,2010.0
1574,Wrigley Field,Chicago,US,NaN,IL,60654.0,NaN,NaN,NaN
1630,Wrigley Field,Chicago,US,NaN,IL,60654.0,NaN,NaN,NaN
1761,Aviva Stadium,Dublin,IE,True,NaN,NaN,Europe/Dublin,2.264277,2010.0
1951,Wembley Stadium,London,GB,True,NaN,NaN,Europe/London,47.000000,2007.0


In [44]:
wrigley = df["venue"] == "Wrigley Field"
df.loc[wrigley, "grass"] = True
df.loc[wrigley, "timezone"] = "America/Chicago"
df.loc[wrigley, "elevation"] = 179.0
df.loc[wrigley, "constructionYear"] = 1914.0

remaining_gaps = df[["grass", "timezone", "elevation", "constructionYear"]].isna().sum()
print(remaining_gaps)
assert (remaining_gaps == 0).all()

# state and zip stay NaN for Aviva Stadium and Wembley Stadium, correct for non-US venues
df[df["state"].isna() | df["zip"].isna()][["venue", "city", "countryCode", "state", "zip"]].drop_duplicates()

grass               0
timezone            0
elevation           0
constructionYear    0
dtype: int64


,venue,city,countryCode,state,zip
0,Aviva Stadium,Dublin,IE,NaN,NaN
1951,Wembley Stadium,London,GB,NaN,NaN


## 6b. Capacity Recorded as Zero Instead of Missing

While sanity checking value ranges, making sure nothing was negative or otherwise impossible, we found 4 rows, all Wrigley Field, all Northwestern's temporary neutral-site home games during their stadium renovation, where `capacity` is literally `0.0` instead of `NaN`. Looking back at the pull script, its label logic already treats `capacity <= 0` as effectively missing internally (`cap = df["capacity"].where(df["capacity"] > 0)`), but that fixed-up version was never saved back into the actual `capacity` column. So the raw CSV was carrying a fake zero that looks like a real value.

Fixing it does not change any of our labels, since these games are already excluded from labeling anyway (`neutralSite = True`). We still fix it so `capacity` does not misrepresent itself to anything reading it directly.

In [45]:
zero_capacity = df["capacity"] <= 0
print(f"{zero_capacity.sum()} rows with capacity <= 0, all flagged missing already:",
      df.loc[zero_capacity, "flag_missing_capacity"].all())

df.loc[zero_capacity, "capacity"] = pd.NA
assert (df["capacity"].dropna() > 0).all()

4 rows with capacity <= 0, all flagged missing already: True


## 7. Final Validation and Export

Before saving, we run one last sanity check: every column should now be completely filled in, except for the ones we deliberately decided to leave blank. Anything outside that list showing up as missing here would mean we missed something.

The columns we are okay leaving as `NaN`, and why:
- `attendance`, `attendance_pct`, `high_attendance`: our label. We cannot fill in a target we do not actually know (section 3).
- `capacity`: just the 4 sentinel-zero rows we fixed in section 6b. We do not know Wrigley's real football-configuration capacity, so we chose not to guess.
- `state`, `zip`: the 2 non-US venues. Correctly blank, not a gap.
- `homePostgameWinProbability`, `awayPostgameWinProbability`, `excitementIndex`: the one game from section 5c with no other row to fill it from.

If the check finds anything else in that list, the `assert` will fail loudly instead of us finding out later that something slipped through.

In [46]:
allowed_missing = {
    "attendance", "attendance_pct", "high_attendance",
    "capacity", "state", "zip",
    "homePostgameWinProbability", "awayPostgameWinProbability", "excitementIndex",
}
still_missing = df.isna().sum()
unexpected = still_missing[(still_missing > 0) & ~still_missing.index.isin(allowed_missing)]

print(still_missing[still_missing > 0])
assert unexpected.empty, f"unexpected NaNs remain in: {list(unexpected.index)}"

attendance                    286
homePostgameWinProbability      1
awayPostgameWinProbability      1
excitementIndex                 1
capacity                        4
state                           4
zip                             4
attendance_pct                290
high_attendance               348
dtype: int64


In [47]:
df.to_csv("../data/full_dataset_clean.csv", index=False)
print("Saved data/full_dataset_clean.csv", df.shape)

Saved data/full_dataset_clean.csv (2021, 51)
